This notebook implements a simple neural network with backpropagation from scratch. We start by importing the necessary libraries: `numpy` for numerical operations and `pandas` for data handling.

In [1]:
import numpy as np
import pandas as pd

Here, we create a small DataFrame `df` which serves as our dataset. It contains `cgpa`, `profile_score` as features, and `placed` as the target variable (0 or 1).

In [2]:
df = pd.DataFrame([[8,8,1],[7,9,1],[6,10,0],[5,5,0]], columns=['cgpa', 'profile_score', 'placed'])

In [3]:
df.head()

,cgpa,profile_score,placed
0,8,8,1
1,7,9,1
2,6,10,0
3,5,5,0


## **Step 1 : Initialize Parameters**
The `initialize_parameters` function sets up the weight matrices (W) and bias vectors (b) for each layer of the neural network. Weights are initialized to small values (0.1) and biases to zeros. `layer_dims` defines the number of neurons in each layer (e.g., `[2, 2, 1]` means 2 input features, 2 neurons in the hidden layer, and 1 output neuron).

In [4]:
def initialize_parameters(layer_dims):

  np.random.seed(3)
  parameters = {}
  L = len(layer_dims)

  for l in range(1, L):

    parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l]))*0.1
    parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))


  return parameters

The `sigmoid` function is a common activation function used in neural networks, especially for binary classification problems. It squashes the input values to a range between 0 and 1.

In [7]:
# Utility Functions
def sigmoid(Z):
  A = 1/(1+np.exp(-Z))
  return A

The `linear_forward` function calculates the linear combination of inputs and weights (`Z = W.T * A_prev + b`) and then applies the `sigmoid` activation function to produce the output `A` for a single layer.

In [6]:
def linear_forward(A_prev, W, b):
  Z = np.dot(W.T, A_prev) + b
  A = sigmoid(Z)
  return A

## **Step 2 : Forward Propagation**
The `L_layer_forward` function performs the forward propagation for the entire neural network. It iterates through each layer, applying the `linear_forward` function to calculate the activations of successive layers until the final output `A` (y_hat) is obtained. It also returns the activation of the previous layer (A1 in this context) which is useful for backpropagation.

In [8]:
# L-layer feed forward

def L_layer_forward(X, parameters):

  A = X
  L = len(parameters) // 2

  for l in range(1, L+1):
    A_prev = A
    Wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]

    A = linear_forward(A_prev, Wl, bl)

  return A,A_prev


## **Step 3 : Weights and Bias Updation**
The `update_parameters` function implements the backpropagation algorithm to adjust the weights and biases of the neural network. It uses a learning rate (0.0001) to update parameters based on the difference between the predicted output (`y_hat`) and the actual output (`y`), propagating the error backward through the layers. This is a manual implementation of gradient descent for a 2-layer network.

In [9]:
def update_parameters(parameters,y,y_hat,A1,X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.0001 * (y - y_hat)*A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.0001 * (y - y_hat)*A1[1][0])
  parameters['b2'][0][0] = parameters['W2'][1][0] + (0.0001 * (y - y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.0001 * (y - y_hat)*parameters['W2'][0][0]*A1[0][0]*(1-A1[0][0])*X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.0001 * (y - y_hat)*parameters['W2'][0][0]*A1[0][0]*(1-A1[0][0])*X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.0001 * (y - y_hat)*parameters['W2'][0][0]*A1[0][0]*(1-A1[0][0]))

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.0001 * (y - y_hat)*parameters['W2'][1][0]*A1[1][0]*(1-A1[1][0])*X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.0001 * (y - y_hat)*parameters['W2'][1][0]*A1[1][0]*(1-A1[1][0])*X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.0001 * (y - y_hat)*parameters['W2'][1][0]*A1[1][0]*(1-A1[1][0]))

This block implements the training loop (epochs). It initializes the network parameters, then iterates for a specified number of `epochs`. In each epoch, it iterates through every training example in the DataFrame:
1. It reshapes the input features `X` and gets the true label `y`.
2. It performs forward propagation (`L_layer_forward`) to get the predicted output `y_hat` and the hidden layer activation `A1`.
3. It updates the network parameters using `update_parameters` based on the prediction error.
4. It calculates the binary cross-entropy loss for each example and prints the average loss for the epoch. The final `parameters` after training are displayed.

In [10]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 50

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):
    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['placed']].values[j][0]

    # Parameter initialization
    y_hat,A1 = L_layer_forward(X,parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters,y,y_hat,A1,X)
    Loss.append(-y*np.log(y_hat) - (1-y)*np.log(1-y_hat))

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

Epoch -  1 Loss -  0.7103199085929446
Epoch -  2 Loss -  0.6991702892802629
Epoch -  3 Loss -  0.6991679314811485
Epoch -  4 Loss -  0.6991655746710999
Epoch -  5 Loss -  0.6991632188496667
Epoch -  6 Loss -  0.699160864016399
Epoch -  7 Loss -  0.6991585101708473
Epoch -  8 Loss -  0.6991561573125619
Epoch -  9 Loss -  0.6991538054410936
Epoch -  10 Loss -  0.6991514545559935
Epoch -  11 Loss -  0.6991491046568126
Epoch -  12 Loss -  0.6991467557431024
Epoch -  13 Loss -  0.6991444078144144
Epoch -  14 Loss -  0.6991420608703007
Epoch -  15 Loss -  0.6991397149103132
Epoch -  16 Loss -  0.6991373699340042
Epoch -  17 Loss -  0.6991350259409265
Epoch -  18 Loss -  0.6991326829306324
Epoch -  19 Loss -  0.6991303409026751
Epoch -  20 Loss -  0.699127999856608
Epoch -  21 Loss -  0.6991256597919842
Epoch -  22 Loss -  0.6991233207083575
Epoch -  23 Loss -  0.6991209826052818
Epoch -  24 Loss -  0.699118645482311
Epoch -  25 Loss -  0.6991163093389996
Epoch -  26 Loss -  0.699113974174902

{'W1': array([[0.09994267, 0.09984548],
        [0.09994272, 0.09984548]]),
 'b1': array([[-3.38405750e-05],
        [-3.38419977e-05]]),
 'W2': array([[0.09920806],
        [0.09920816]]),
 'b2': array([[0.09915209]])}